In [0]:
# MERGE INTO (upsert) — Databricks SQL / Delta Lake
#
# MERGE INTO lets you update, insert, and even delete rows in a target Delta table
# in a single atomic statement, based on matching rows in a source table. It's the
# standard way to do upserts ("update if the key exists, insert if it doesn't").
#
# Syntax:
#   MERGE INTO target AS t
#   USING source AS s
#   ON t.key = s.key            -- join/match condition
#   WHEN MATCHED THEN UPDATE SET t.col = s.col, ...
#   WHEN NOT MATCHED THEN INSERT (cols) VALUES (vals)
#   WHEN NOT MATCHED BY SOURCE THEN DELETE   -- optional: delete target rows
#                                             -- with no source match
#
# How it works:
#  - WHEN MATCHED:              source and target rows match on the ON condition
#                               -> typically UPDATE (or DELETE for SCD removal)
#  - WHEN NOT MATCHED:          source row has no target match -> INSERT
#  - WHEN NOT MATCHED BY SOURCE: target row has no source match -> DELETE or UPDATE
#  - Multiple matched clauses can carry extra AND <condition> predicates, e.g.
#      WHEN MATCHED AND s.amount <> t.amount THEN UPDATE ...
#
# Example: upsert daily customer updates into a customers table

spark.sql("""
MERGE INTO catalog.schema.customers AS t
USING catalog.schema.daily_updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.address <> s.address THEN UPDATE SET
  t.address = s.address,
  t.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (customer_id, address, updated_at)
VALUES
  (s.customer_id, s.address, current_timestamp())
""")

# Tips for Databricks:
#  - For CDC, Delta's change data feed is a natural MERGE source (keyed by the
#    primary key, ordered by a sequence/timestamp column).
#  - SCD Type 2: use MERGE to close old rows (update end_date) and insert new ones.
#  - Only one target row should match any given source row, otherwise MERGE fails
#    with a MultipleSourceRowsMatchingException — dedupe the source first.
#  - For large merges, partition-prune the target in the ON condition
#    (e.g. include a date column) to avoid full-table scans.
#  - Streaming tables / Lakeflow pipelines have built-in equivalents — a plain
#    MERGE is for batch jobs and notebooks like this one.


In [0]:
# INTERVIEW Q&A: MERGE INTO in Databricks / Delta Lake
# (interview-ready answers based on the concepts above)

# ------------------------------------------------------------------
# Q1. What is MERGE INTO and when would you use it?
# A. MERGE INTO (upsert) atomically applies UPDATE, INSERT, and DELETE
#    to a target Delta table based on matching rows from a source.
#    Use it for incremental loads, CDC ingestion, and SCD Type 2 history:
#    matching rows get updated, new keys get inserted, and (optionally)
#    target rows with no source match get deleted — all in ONE atomic
#    statement, so a failure never leaves the table half-updated.
#
# Q2. What does each WHEN clause do?
# A.  - WHEN MATCHED                  -> source row found in target (UPDATE/DELETE)
#    - WHEN NOT MATCHED               -> source row is new (INSERT)
#    - WHEN NOT MATCHED BY SOURCE     -> target row has no source match (DELETE/UPDATE)
#    Each clause can carry extra predicates, e.g.
#    WHEN MATCHED AND s.amount <> t.amount THEN UPDATE ...
#
# Q3. Why is MERGE better than DELETE + INSERT or separate UPDATE/INSERT?
# A. Atomicity (one transaction, no partial state), performance (single
#    pass instead of multiple scans/jobs), and simplicity (one statement
#    captures all upsert logic). It also preserves table history/statistics
#    without rewriting the whole table.
#
# Q4. What is MultipleSourceRowsMatchingException and how do you fix it?
# A. MERGE requires at most ONE target row per source row per clause.
#    If the source has duplicate keys, the merge fails. Fix: dedupe the
#    source first (e.g. row_number() over the key ordered by a version/
#    timestamp column) so each key appears exactly once.
#
# Q5. How do you implement SCD Type 2 with MERGE?
# A. Two-part pattern:
#    1) WHEN MATCHED AND value changed -> UPDATE SET end_date = current_date(), is_current = false
#    2) WHEN NOT MATCHED -> INSERT the new version with start_date = today, is_current = true
#    Delta's change data feed is a natural MERGE source for CDC streams,
#    keyed by primary key and ordered by a sequence/timestamp column.
#
# Q6. How do you make large MERGEs fast?
# A.  - Partition-prune the target via the ON condition (e.g. include a
#       date column) so only relevant files are touched — otherwise MERGE
#       full-scans the table.
#    - Liquid clustering / Z-ORDER on the join key.
#    - Dedupe the source to avoid skew and multiple-match errors.
#
# Q7. What are MERGE's limitations in Databricks?
# A.  - Batch-oriented: for streaming upserts, prefer streaming tables /
#       Lakeflow pipelines or CREATE STREAMING TABLE ... FLOW AUTO CDC,
#       which have built-in equivalents.
#    - Only one source match per target row allowed per clause.
#    - ON condition should target the data skipping / partition layout.
# ------------------------------------------------------------------


In [0]:
spark.sql("DROP TABLE IF EXISTS source_pnd")
spark.sql("DROP TABLE IF EXISTS target_pnd")

In [0]:
%sql
CREATE TABLE workspace.default.source_pnd(
    id INT,
    name STRING,
    city STRING
)
using DELTA;

In [0]:
%sql
INSERT INTO workspace.default.source_pnd 
VALUES (1, 'John', 'NYC'),
       (2, 'Jane', 'LA'), 
       (3, 'Bob', 'NYC'), 
       (4, 'Alice', 'LA');

In [0]:
%sql
CREATE TABLE target_pnd (
    id INT,
    name STRING,
    city STRING
)
using DELTA;


In [0]:
%sql
MERGE INTO target_pnd t 
USING source_pnd s
ON t.id = s.id 
WHEN MATCHED THEN 
UPDATE SET t.name = s.name,
           t.city = s.city
WHEN NOT MATCHED THEN 
INSERT (id, name, city) 
VALUES (s.id, s.name, s.city);

In [0]:
%sql
SELECT * FROM target_pnd;

In [0]:
%sql
INSERT INTO workspace.default.source_pnd
VALUES (5, 'Mike', 'NYC');
    
MERGE INTO target_pnd t 
USING source_pnd s
ON t.id = s.id 
WHEN MATCHED THEN 
UPDATE SET t.name = s.name,
           t.city = s.city
WHEN NOT MATCHED THEN 
INSERT (id, name, city) 
VALUES (s.id, s.name, s.city)

In [0]:
%sql
UPDATE workspace.default.source_pnd
SET name = 'Mary'
WHERE id = 2;
    

In [0]:
display(spark.sql("select * from source_pnd"))

In [0]:
display(spark.sql("select * from target_pnd"))

In [0]:
%sql
MERGE INTO target_pnd t 
USING source_pnd s 
ON t.id = s.id 
WHEN MATCHED THEN 
UPDATE SET t.name = s.name,
           t.city = s.city
WHEN NOT MATCHED THEN 
INSERT (id, name, city) 
VALUES (s.id, s.name, s.city);
    
SELECT * FROM target_pnd;
    

In [0]:
# FULL LOAD vs INCREMENTAL LOAD
#
# ------------------------------------------------------------------
# FULL LOAD (a.k.a. complete / full refresh)
# ------------------------------------------------------------------
#  - The ENTIRE dataset from the source is loaded into the target every run.
#  - Target is typically truncated/overwritten first, then reloaded:
#      TRUNCATE TABLE target;  INSERT INTO target SELECT * FROM source;
#    or in Spark: df.write.mode("overwrite").saveAsTable("target")
#  - Simple to implement and reason about: no watermark/offset tracking.
#  - Captures ALL changes, including updates, deletes, and late-arriving
#    corrections, because the whole snapshot replaces the old state.
#  - Downsides: expensive and slow for big tables (full scan + full rewrite
#    every run), heavy on network/compute, and you lose row-level history
#    unless you keep separate snapshots.
#  - Good for: small dimension tables, reference data, or full reprocessing
#    when the source can't provide change metadata.
#
# ------------------------------------------------------------------
# INCREMENTAL LOAD (a.k.a. delta load)
# ------------------------------------------------------------------
#  - Only rows that are NEW or CHANGED since the last run are loaded.
#  - Requires a way to identify "what's new" since the last load:
#      * a high-watermark column (updated_at / last_modified timestamp)
#      * a monotonically increasing id / sequence number
#      * a CDC source (Change Data Feed, Debezium, etc.) that emits
#        insert/update/delete events
#  - Two common flavors:
#      1) APPEND-ONLY: new rows inserted, existing rows never change
#         -> WHERE updated_at > :last_watermark, then INSERT
#      2) UPSERT: rows can be inserted OR updated
#         -> MERGE INTO target t USING source_delta s ON t.id = s.id
#            WHEN MATCHED THEN UPDATE SET ...
#            WHEN NOT MATCHED THEN INSERT ...
#         (This is exactly the MERGE INTO pattern used in this notebook.)
#  - Upsides: fast, cheap, scalable — moves only the delta, so it's the
#    standard choice for large fact/event tables and daily pipelines.
#  - Downsides: more complex — you must reliably track the watermark/offset,
#    handle late-arriving or out-of-order data, and detect deletes (an
#    append-only incremental load can't see a row deleted in the source).
#  - Good for: large, growing event/transaction tables and CDC pipelines.
#
# ------------------------------------------------------------------
# SIDE-BY-SIDE
# ------------------------------------------------------------------
#  | Aspect            | Full load                  | Incremental load         |
#  |-------------------|----------------------------|---------------------------|
#  | Data moved        | Everything, every run      | Only new/changed rows     |
#  | Implementation    | Trivial (overwrite)        | Watermark/CDC + MERGE     |
#  | Runtime & cost    | High, grows with table     | Low, grows with the delta |
#  | Deletes captured  | Yes (fresh snapshot)       | Only if source is CDC     |
#  | History kept      | No (unless snapshotted)    | Yes (with SCD2/CDF)       |
#  | Failure recovery  | Just rerun the full load   | Watermark must be managed |
#
# In Databricks, incremental loading with Delta Lake is the default
# recommendation: use MERGE INTO for batch upserts (as shown above), or
# streaming tables / Lakeflow pipelines with Change Data Feed for
# continuous ingestion. Full loads are mainly a fallback for small tables
# or sources that cannot expose change metadata.
# ------------------------------------------------------------------

In [0]:
# SLOWLY CHANGING DIMENSIONS (SCD) — TYPE 1 vs TYPE 2
# A "slowly changing dimension" is a table like CUSTOMERS where attributes
# (address, phone, marital status...) change occasionally, and you must
# decide HOW to represent that change in your warehouse.
#
# ------------------------------------------------------------------
# SCD TYPE 1 — "overwrite, forget the past"
#  - The old value is simply REPLACED by the new one.
#  - No history kept: the table always shows the LATEST state.
#  - Easiest to implement (a plain UPDATE / upsert MERGE).
#
# SCD TYPE 2 — "keep the full history"
#  - A change does NOT overwrite; instead the old row is CLOSED
#    (is_current = false, valid_to = today) and a NEW row is inserted
#    with the new values (is_current = true, valid_from = today).
#  - Every version of the customer lives in the table, so you can
#    reconstruct what the customer looked like at any point in time
#    ("what address did John have when he placed his 2023 orders?").
#  - Typical extra columns: valid_from, valid_to, is_current
#    (and often a surrogate key + change reason / hash).
#
# | Aspect        | Type 1              | Type 2                        |
# |---------------|---------------------|-------------------------------|
# | History       | None (overwritten)  | Full row versions             |
# | Table size    | 1 row per customer  | 1 row per version             |
# | Implementation| Simple UPDATE/MERGE | Close old + insert new        |
# | Typical use   | Reference/typos     | Addresses, pricing, contracts |
# ------------------------------------------------------------------

# Clean up any previous run of this demo
spark.sql("DROP TABLE IF EXISTS customer_updates")
spark.sql("DROP TABLE IF EXISTS dim_customer_scd1")
spark.sql("DROP TABLE IF EXISTS dim_customer_scd2")

# Source dimension (current view of customers) and a batch of updates:
# John moves from NYC to Boston, and a brand-new customer (id 3) arrives.
spark.sql("CREATE TABLE customer_updates (customer_id INT, name STRING, city STRING)")

spark.sql("""
INSERT INTO customer_updates VALUES
  (1, 'John', 'Boston'),   -- UPDATE: John's city changed
  (3, 'Carol', 'Austin')   -- INSERT: brand-new customer
""")

# ==================================================================
# SCD TYPE 1 — overwrite in place with MERGE
# ==================================================================
spark.sql("CREATE TABLE dim_customer_scd1 (customer_id INT, name STRING, city STRING) USING DELTA")

spark.sql("INSERT INTO dim_customer_scd1 VALUES (1, 'John', 'NYC'), (2, 'Jane', 'LA')")

spark.sql("""
MERGE INTO dim_customer_scd1 AS t
USING customer_updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET t.city = s.city   -- old value is lost
WHEN NOT MATCHED THEN INSERT VALUES (s.customer_id, s.name, s.city)
""")

# After this MERGE there is still ONE row for John -> city = 'Boston'.
# 'NYC' is gone forever: that is the whole point (and the limitation) of Type 1.
display(spark.sql("SELECT * FROM dim_customer_scd1 ORDER BY customer_id"))

# ==================================================================
# SCD TYPE 2 — keep every version using valid_from/valid_to/is_current
# ==================================================================
spark.sql("""
CREATE TABLE dim_customer_scd2 (
  customer_id INT,
  name STRING,
  city STRING,
  valid_from DATE,
  valid_to   DATE,
  is_current BOOLEAN
) USING DELTA
""")

spark.sql("""
INSERT INTO dim_customer_scd2 VALUES
  (1, 'John', 'NYC', DATE'2020-01-01', DATE'9999-12-31', true),
  (2, 'Jane', 'LA',  DATE'2020-01-01', DATE'9999-12-31', true)
""")

# Step 1: CLOSE the old version of any customer whose attributes changed
# (only current rows can be closed; unchanged customers are untouched).
spark.sql("""
MERGE INTO dim_customer_scd2 AS t
USING customer_updates AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.is_current AND t.city <> s.city THEN
  UPDATE SET t.is_current = false,
             t.valid_to = current_date()
""")

# Step 2: INSERT a fresh version for every updated customer,
#         plus entirely new customers.
# The source is the updates left with NO current row (either closed in
# step 1, or never seen before).
spark.sql("""
MERGE INTO dim_customer_scd2 AS t
USING (
  SELECT s.*
  FROM customer_updates s
  LEFT JOIN dim_customer_scd2 c
    ON c.customer_id = s.customer_id AND c.is_current
  WHERE c.customer_id IS NULL
) AS s
ON false
WHEN NOT MATCHED THEN INSERT
  (customer_id, name, city, valid_from, valid_to, is_current)
VALUES
  (s.customer_id, s.name, s.city, current_date(), DATE'9999-12-31', true)
""")

# John now has TWO rows: the NYC version (closed) and the Boston version
# (current). Jane is unchanged, and Carol arrived as a new current row.
display(spark.sql("SELECT * FROM dim_customer_scd2 ORDER BY customer_id, valid_from"))

# Point-in-time query: what did each customer look like on 2023-06-30?
# -> John shows 'NYC', because that's the version valid on that date.
display(spark.sql("""
SELECT customer_id, name, city
FROM dim_customer_scd2
WHERE DATE'2023-06-30' >= valid_from
  AND DATE'2023-06-30' <  valid_to
ORDER BY customer_id
"""))

# Interview notes:
#  - Type 1 = MERGE ... WHEN MATCHED THEN UPDATE (in-place overwrite).
#  - Type 2 = close the old row (UPDATE) + insert the new version — the
#    two-step pattern above, or one MERGE with a UNION source trick.
#  - Facts should join to the dimension version valid on the fact's date
#    (fact_date BETWEEN valid_from AND valid_to) — that's the payoff of SCD2.
#  - In Databricks, streaming CDC pipelines (FLOW AUTO CDC with SCD TYPE 2)
#    can maintain Type 2 tables for you automatically.


In [0]:
%sql
SELECT * FROM customer_updates;


In [0]:
%python
# ==================================================================
# SCD TYPE 2 — keep every version using valid_from/valid_to/is_current
# ==================================================================
spark.sql("""
CREATE TABLE dim_customer_scd2 (
  customer_id INT,
  name STRING,
  city STRING,
  valid_from DATE,
  valid_to   DATE,
  is_current BOOLEAN
) USING DELTA
""")
    


In [0]:
%sql
MERGE INTO dim_customer_scd2 t
USING customer_updates s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.is_current = true THEN
UPDATE SET
  t.name = s.name,
  t.city = s.city,
  t.valid_to = current_date()
WHEN NOT MATCHED THEN
INSERT (
  customer_id,
  name,
  city,
  valid_from,
  valid_to,
  is_current
) 
VALUES
  (s.customer_id, s.name, s.city, current_date(), '9999-12-31', true)


In [0]:
display(spark.sql("SELECT * FROM dim_customer_scd2"))

In [0]:
%sql

INSERT INTO customer_updates
VALUES (2, 'Ajay', 'New York');

UPDATE customer_updates
SET name = 'Mary'
WHERE customer_id =3;


SELECT * FROM customer_updates;



In [0]:
%sql
INSERT INTO customer_updates
VALUES (2, 'Ajay', 'New York');

UPDATE customer_updates
SET name = 'Mary'
WHERE customer_id =3

In [0]:
%sql
SELECT * FROM customer_updates;

In [0]:
%sql
SELECT * FROM dim_customer_scd2;

In [0]:
%sql
MERGE INTO dim_customer_scd2 t
USING customer_updates s
ON t.customer_id = s.customer_id
AND t.is_current = true
WHEN MATCHED AND (t.name != s.name OR t.city != s.city)  
THEN
UPDATE SET
  t.name = s.name,
  t.city = s.city,
  t.is_current = false,
  t.valid_to = current_date()
WHEN NOT MATCHED THEN
INSERT (customer_id, name, city, valid_from, valid_to, is_current)
VALUES (s.customer_id, s.name, s.city, current_date(), DATE '9999-12-31', true);
    
SELECT * FROM dim_customer_scd2;

